In [1]:
# === Cellule 1 : Import des librairies et chargement du CSV ===
import pandas as pd
import numpy as np
from scipy.stats.mstats import winsorize
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.metrics import mean_squared_error
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

# Charger le fichier CSV
df = pd.read_csv(r"C:\Users\zizou\OneDrive\Desktop\stage 3ème\day 2\csvfiles\dataframefinale.csv", sep=';')
print("Aperçu des données :")
display(df.head())


Aperçu des données :


,dataloadingdate,Jour,Mois,NumeroSemaine,Trimestre,JourSemaineNum,JourSemaine,ISIN,Libellé,Nombre de Titres,Montant,Echéance,Taux
0,01/07/2025,1,7,27,3,1,Tuesday,TN0008000739,"BTA 7,4% Fevrier 2030",1459.0,1.500,62.0,7.6
1,01/07/2025,1,7,27,3,1,Tuesday,TN0008000739,"BTA 7,4% Fevrier 2030",291.0,0.300,183.0,7.5
2,01/07/2025,1,7,27,3,1,Tuesday,TNMCPXLL1EE2,EMP NAT 2023 T4 CB TV,50000.0,5.000,13.0,8.6
3,01/07/2025,1,7,27,3,1,Tuesday,TNMCPXLL1EE2,EMP NAT 2023 T4 CB TV,10000.0,1.000,7.0,8.6
4,01/07/2025,1,7,27,3,1,Tuesday,TN0008000606,"BTA 6,7% Avril 2028",5660.0,5.742,31.0,9.1


In [2]:
# === Cellule 2 : Winsorisation de toutes les colonnes numériques et log-transform de la target ===

# Sélection de toutes les colonnes numériques
numeric_cols = df.select_dtypes(include=['int64', 'float64']).columns.tolist()

# Copie du dataframe pour la winsorisation
df_winsor = df.copy()

# Winsorisation 5%-95% sur toutes les colonnes numériques
for col in numeric_cols:
    df_winsor[col] = winsorize(df[col], limits=[0.05, 0.05])

# Log-transform de la target 'Montant'
df_winsor['Montant_log'] = np.log1p(df_winsor['Montant'])

# Vérification
print("Statistiques après winsorisation et log-transform :")
display(df_winsor[numeric_cols + ['Montant_log']].describe())


Statistiques après winsorisation et log-transform :


,Jour,Mois,NumeroSemaine,Trimestre,JourSemaineNum,Nombre de Titres,Montant,Echéance,Taux,Montant_log
count,20267.000000,20267.000000,20267.000000,20267.000000,20267.000000,20267.000000,20267.000000,20267.000000,20267.000000,20267.000000
mean,14.850200,6.600730,26.628509,2.550155,1.954409,7597.774165,4.133039,33.962599,8.627887,1.307947
std,9.057001,3.394496,14.508804,1.103134,1.474442,10679.660190,4.635870,36.140342,0.759614,0.783107
min,1.000000,1.000000,3.000000,1.000000,0.000000,281.000000,0.225000,6.000000,7.250000,0.202941
25%,7.000000,4.000000,14.000000,2.000000,1.000000,1078.000000,1.000000,10.000000,8.030000,0.693147
50%,14.000000,7.000000,27.000000,3.000000,2.000000,3180.000000,2.001000,21.000000,8.970000,1.098946
75%,23.000000,10.000000,40.000000,4.000000,3.000000,8986.500000,5.500000,32.000000,9.010000,1.871802
max,30.000000,12.000000,49.000000,4.000000,4.000000,41689.000000,18.000000,145.000000,9.890000,2.944439


In [3]:
# === Cellule 3 : Enregistrement de la DataFrame winsorisée ===
output_path = r"C:\Users\zizou\OneDrive\Desktop\stage 3ème\day 2\csvfiles\dataframefinale_winsor.csv"
df_winsor.to_csv(output_path, index=False)
print(f"La DataFrame winsorisée a été enregistrée dans : {output_path}")


La DataFrame winsorisée a été enregistrée dans : C:\Users\zizou\OneDrive\Desktop\stage 3ème\day 2\csvfiles\dataframefinale_winsor.csv


In [4]:
# === Cellule 4 : Comparaison DataFrame initiale vs winsorisée ===

print("=== Infos du DataFrame initial ===")
display(df.info())
print("\n=== Statistiques descriptives du DataFrame initial ===")
display(df.describe())

print("\n=== Infos du DataFrame winsorisé ===")
display(df_winsor.info())
print("\n=== Statistiques descriptives du DataFrame winsorisé ===")
display(df_winsor.describe())

# Optionnel : comparaison rapide des valeurs min/max pour voir l'effet de la winsorisation
min_max_compare = pd.DataFrame({
    'Initial_min': df[numeric_cols].min(),
    'Initial_max': df[numeric_cols].max(),
    'Winsor_min': df_winsor[numeric_cols].min(),
    'Winsor_max': df_winsor[numeric_cols].max()
})
print("\n=== Comparaison min/max des colonnes numériques ===")
display(min_max_compare)


=== Infos du DataFrame initial ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20267 entries, 0 to 20266
Data columns (total 13 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   dataloadingdate   20267 non-null  object 
 1   Jour              20267 non-null  int64  
 2   Mois              20267 non-null  int64  
 3   NumeroSemaine     20267 non-null  int64  
 4   Trimestre         20267 non-null  int64  
 5   JourSemaineNum    20267 non-null  int64  
 6   JourSemaine       20267 non-null  object 
 7   ISIN              20267 non-null  object 
 8   Libellé           20267 non-null  object 
 9   Nombre de Titres  20267 non-null  float64
 10  Montant           20267 non-null  float64
 11  Echéance          20267 non-null  float64
 12  Taux              20267 non-null  float64
dtypes: float64(4), int64(5), object(4)
memory usage: 2.0+ MB


None


=== Statistiques descriptives du DataFrame initial ===


,Jour,Mois,NumeroSemaine,Trimestre,JourSemaineNum,Nombre de Titres,Montant,Echéance,Taux
count,20267.000000,20267.000000,20267.000000,20267.000000,20267.000000,20267.000000,20267.000000,20267.000000,20267.000000
mean,14.872107,6.600730,26.662012,2.550155,1.955988,9106.529679,4.849259,40.061627,8.620955
std,9.094752,3.394496,14.783995,1.103134,1.477165,17518.614949,7.943545,60.836203,0.809927
min,1.000000,1.000000,1.000000,1.000000,0.000000,1.000000,0.001000,0.000000,5.800000
25%,7.000000,4.000000,14.000000,2.000000,1.000000,1078.000000,1.000000,10.000000,8.030000
50%,14.000000,7.000000,27.000000,3.000000,2.000000,3180.000000,2.001000,21.000000,8.970000
75%,23.000000,10.000000,40.000000,4.000000,3.000000,8986.500000,5.500000,32.000000,9.010000
max,31.000000,12.000000,52.000000,4.000000,5.000000,200000.000000,100.000000,365.000000,10.750000



=== Infos du DataFrame winsorisé ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20267 entries, 0 to 20266
Data columns (total 14 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   dataloadingdate   20267 non-null  object 
 1   Jour              20267 non-null  int64  
 2   Mois              20267 non-null  int64  
 3   NumeroSemaine     20267 non-null  int64  
 4   Trimestre         20267 non-null  int64  
 5   JourSemaineNum    20267 non-null  int64  
 6   JourSemaine       20267 non-null  object 
 7   ISIN              20267 non-null  object 
 8   Libellé           20267 non-null  object 
 9   Nombre de Titres  20267 non-null  float64
 10  Montant           20267 non-null  float64
 11  Echéance          20267 non-null  float64
 12  Taux              20267 non-null  float64
 13  Montant_log       20267 non-null  float64
dtypes: float64(5), int64(5), object(4)
memory usage: 2.2+ MB


...


=== Statistiques descriptives du DataFrame winsorisé ===


c:\Users\zizou\anaconda3\Lib\site-packages\numpy\lib\function_base.py:4824: UserWarning: Warning: 'partition' will ignore the 'mask' of the MaskedArray.
  arr.partition(
c:\Users\zizou\anaconda3\Lib\site-packages\numpy\lib\function_base.py:4824: UserWarning: Warning: 'partition' will ignore the 'mask' of the MaskedArray.
  arr.partition(
c:\Users\zizou\anaconda3\Lib\site-packages\numpy\lib\function_base.py:4824: UserWarning: Warning: 'partition' will ignore the 'mask' of the MaskedArray.
  arr.partition(
c:\Users\zizou\anaconda3\Lib\site-packages\numpy\lib\function_base.py:4824: UserWarning: Warning: 'partition' will ignore the 'mask' of the MaskedArray.
  arr.partition(
c:\Users\zizou\anaconda3\Lib\site-packages\numpy\lib\function_base.py:4824: UserWarning: Warning: 'partition' will ignore the 'mask' of the MaskedArray.
  arr.partition(
c:\Users\zizou\anaconda3\Lib\site-packages\numpy\lib\function_base.py:4824: UserWarning: Warning: 'partition' will ignore the 'mask' of the MaskedArra

,Jour,Mois,NumeroSemaine,Trimestre,JourSemaineNum,Nombre de Titres,Montant,Echéance,Taux,Montant_log
count,20267.000000,20267.000000,20267.000000,20267.000000,20267.000000,20267.000000,20267.000000,20267.000000,20267.000000,20267.000000
mean,14.850200,6.600730,26.628509,2.550155,1.954409,7597.774165,4.133039,33.962599,8.627887,1.307947
std,9.057001,3.394496,14.508804,1.103134,1.474442,10679.660190,4.635870,36.140342,0.759614,0.783107
min,1.000000,1.000000,3.000000,1.000000,0.000000,281.000000,0.225000,6.000000,7.250000,0.202941
25%,7.000000,4.000000,14.000000,2.000000,1.000000,1078.000000,1.000000,10.000000,8.030000,0.693147
50%,14.000000,7.000000,27.000000,3.000000,2.000000,3180.000000,2.001000,21.000000,8.970000,1.098946
75%,23.000000,10.000000,40.000000,4.000000,3.000000,8986.500000,5.500000,32.000000,9.010000,1.871802
max,30.000000,12.000000,49.000000,4.000000,4.000000,41689.000000,18.000000,145.000000,9.890000,2.944439



=== Comparaison min/max des colonnes numériques ===


,Initial_min,Initial_max,Winsor_min,Winsor_max
Jour,1.000,31.00,1.000,30.00
Mois,1.000,12.00,1.000,12.00
NumeroSemaine,1.000,52.00,3.000,49.00
Trimestre,1.000,4.00,1.000,4.00
JourSemaineNum,0.000,5.00,0.000,4.00
Nombre de Titres,1.000,200000.00,281.000,41689.00
Montant,0.001,100.00,0.225,18.00
Echéance,0.000,365.00,6.000,145.00
Taux,5.800,10.75,7.250,9.89


In [5]:
# === Cellule 5 : Séparation des features et de la target + train/test split ===

# Features (toutes les colonnes sauf la target originale et log-transform)
X = df_winsor.drop(columns=['Montant', 'Montant_log'])
# Target log-transformée
y = df_winsor['Montant_log']

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Vérification des dimensions
print(f"Dimensions X_train : {X_train.shape}")
print(f"Dimensions X_test  : {X_test.shape}")
print(f"Dimensions y_train : {y_train.shape}")
print(f"Dimensions y_test  : {y_test.shape}")


Dimensions X_train : (16213, 12)
Dimensions X_test  : (4054, 12)
Dimensions y_train : (16213,)
Dimensions y_test  : (4054,)


In [6]:
# === Cellule 6 : Fonction pour entraîner un modèle avec RandomizedSearchCV rapide ===
def train_model(model, param_dist, X_train, y_train, X_test, y_test, n_iter=20, cv=3):
    """
    Entraîne un modèle avec RandomizedSearchCV et retourne :
    - le meilleur modèle
    - le RMSE sur l'échelle originale
    - les meilleurs hyperparamètres
    """
    random_search = RandomizedSearchCV(
        estimator=model,
        param_distributions=param_dist,
        n_iter=n_iter,
        scoring='neg_root_mean_squared_error',
        cv=cv,
        n_jobs=-1,
        random_state=42
    )
    
    # Entraînement
    random_search.fit(X_train, y_train)
    
    # Meilleur modèle
    best_model = random_search.best_estimator_
    
    # Prédiction sur X_test et retour à l'échelle originale
    y_pred_log = best_model.predict(X_test)
    y_pred = np.expm1(y_pred_log)
    
    # RMSE sur l'échelle originale
    rmse = np.sqrt(mean_squared_error(np.expm1(y_test), y_pred))
    
    return best_model, rmse, random_search.best_params_


In [10]:
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
import numpy as np

# --- Sélection uniquement des colonnes numériques ---
X_train_num = X_train.select_dtypes(include=[np.number])
X_test_num  = X_test.select_dtypes(include=[np.number])

# --- 1) RMSE brut ---
ridge_brut = Ridge(random_state=42)
ridge_brut.fit(X_train_num, y_train)
y_pred_brut = np.expm1(ridge_brut.predict(X_test_num))
rmse_brut = np.sqrt(mean_squared_error(np.expm1(y_test), y_pred_brut))

# --- 2) RMSE normalisé ---
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_num)
X_test_scaled  = scaler.transform(X_test_num)

ridge_norm = Ridge(random_state=42)
ridge_norm.fit(X_train_scaled, y_train)
y_pred_norm = np.expm1(ridge_norm.predict(X_test_scaled))
rmse_norm = np.sqrt(mean_squared_error(np.expm1(y_test), y_pred_norm))

# --- 3) RMSE feature selection ---
# Top 6 features corrélées avec la target
numeric_cols_train = X_train_num.columns
top_features = X_train_num.corrwith(y_train).abs().sort_values(ascending=False).index[:6].tolist()

ridge_feat = Ridge(random_state=42)
ridge_feat.fit(X_train_num[top_features], y_train)
y_pred_feat = np.expm1(ridge_feat.predict(X_test_num[top_features]))
rmse_feat = np.sqrt(mean_squared_error(np.expm1(y_test), y_pred_feat))

# --- 4) RMSE fine-tuning rapide ---
def train_model(model, param_grid, X_tr, y_tr, X_te, y_te):
    from sklearn.model_selection import RandomizedSearchCV
    rs = RandomizedSearchCV(
        estimator=model,
        param_distributions=param_grid,
        n_iter=20,
        cv=3,
        scoring='neg_root_mean_squared_error',
        random_state=42,
        n_jobs=-1
    )
    rs.fit(X_tr, y_tr)
    best_model = rs.best_estimator_
    y_pred = np.expm1(best_model.predict(X_te))
    rmse = np.sqrt(mean_squared_error(np.expm1(y_te), y_pred))
    return best_model, rmse, rs.best_params_

ridge_params = {'alpha': np.logspace(-2, 3, 20)}
best_ridge, rmse_finetune, ridge_best_params = train_model(Ridge(random_state=42), ridge_params, X_train_num, y_train, X_test_num, y_test)

# --- Affichage des résultats ---
print("Ridge Regression :")
print(f"RMSE brut           : {rmse_brut:.6f}")
print(f"RMSE normalisé      : {rmse_norm:.6f}")
print(f"RMSE feat select    : {rmse_feat:.6f}")
print(f"RMSE fine-tuning    : {rmse_finetune:.6f}")
print(f"Meilleur alpha fine-tuning : {ridge_best_params['alpha']}")


Ridge Regression :
RMSE brut           : 4.185172
RMSE normalisé      : 4.185085
RMSE feat select    : 4.184598
RMSE fine-tuning    : 4.185883
Meilleur alpha fine-tuning : 162.3776739188721


In [11]:
from sklearn.linear_model import Lasso
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
import numpy as np

# --- Colonnes numériques seulement ---
X_train_num = X_train.select_dtypes(include=[np.number])
X_test_num  = X_test.select_dtypes(include=[np.number])

# --- 1) RMSE brut ---
lasso_brut = Lasso(random_state=42, max_iter=10000)
lasso_brut.fit(X_train_num, y_train)
y_pred_brut = np.expm1(lasso_brut.predict(X_test_num))
rmse_brut = np.sqrt(mean_squared_error(np.expm1(y_test), y_pred_brut))

# --- 2) RMSE normalisé ---
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_num)
X_test_scaled  = scaler.transform(X_test_num)

lasso_norm = Lasso(random_state=42, max_iter=10000)
lasso_norm.fit(X_train_scaled, y_train)
y_pred_norm = np.expm1(lasso_norm.predict(X_test_scaled))
rmse_norm = np.sqrt(mean_squared_error(np.expm1(y_test), y_pred_norm))

# --- 3) RMSE feature selection ---
numeric_cols_train = X_train_num.columns
top_features = X_train_num.corrwith(y_train).abs().sort_values(ascending=False).index[:6].tolist()

lasso_feat = Lasso(random_state=42, max_iter=10000)
lasso_feat.fit(X_train_num[top_features], y_train)
y_pred_feat = np.expm1(lasso_feat.predict(X_test_num[top_features]))
rmse_feat = np.sqrt(mean_squared_error(np.expm1(y_test), y_pred_feat))

# --- 4) RMSE fine-tuning ---
lasso_params = {'alpha': [0.001, 0.01, 0.1, 1, 10, 50, 100]}
best_lasso, rmse_finetune, lasso_best_params = train_model(Lasso(random_state=42, max_iter=10000), lasso_params, X_train_num, y_train, X_test_num, y_test)

# --- Affichage des résultats ---
print("Lasso Regression :")
print(f"RMSE brut           : {rmse_brut:.6f}")
print(f"RMSE normalisé      : {rmse_norm:.6f}")
print(f"RMSE feat select    : {rmse_feat:.6f}")
print(f"RMSE fine-tuning    : {rmse_finetune:.6f}")
print(f"Meilleur alpha fine-tuning : {lasso_best_params['alpha']}")


Lasso Regression :
RMSE brut           : 4.205933
RMSE normalisé      : 4.763527
RMSE feat select    : 4.205933
RMSE fine-tuning    : 4.187016
Meilleur alpha fine-tuning : 0.001


c:\Users\zizou\anaconda3\Lib\site-packages\sklearn\model_selection\_search.py:320: UserWarning: The total space of parameters 7 is smaller than n_iter=20. Running 7 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


In [12]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
import numpy as np

# --- Colonnes numériques seulement ---
X_train_num = X_train.select_dtypes(include=[np.number])
X_test_num  = X_test.select_dtypes(include=[np.number])

# --- 1) RMSE brut ---
rf_brut = RandomForestRegressor(n_estimators=100, random_state=42)
rf_brut.fit(X_train_num, y_train)
y_pred_brut = np.expm1(rf_brut.predict(X_test_num))
rmse_brut = np.sqrt(mean_squared_error(np.expm1(y_test), y_pred_brut))

# --- 2) RMSE normalisé (pas nécessaire mais pour homogénéité) ---
rf_norm = RandomForestRegressor(n_estimators=100, random_state=42)
rf_norm.fit(X_train_num, y_train)  # mêmes données, RF n'a pas besoin de scaler
y_pred_norm = np.expm1(rf_norm.predict(X_test_num))
rmse_norm = np.sqrt(mean_squared_error(np.expm1(y_test), y_pred_norm))

# --- 3) RMSE feature selection ---
numeric_cols_train = X_train_num.columns
top_features = X_train_num.corrwith(y_train).abs().sort_values(ascending=False).index[:6].tolist()

rf_feat = RandomForestRegressor(n_estimators=100, random_state=42)
rf_feat.fit(X_train_num[top_features], y_train)
y_pred_feat = np.expm1(rf_feat.predict(X_test_num[top_features]))
rmse_feat = np.sqrt(mean_squared_error(np.expm1(y_test), y_pred_feat))

# --- 4) RMSE fine-tuning rapide ---
rf_params = {
    'n_estimators': [100, 200, 300, 400, 500],
    'max_depth': [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': [0.6, 0.8, 1.0, None]
}

best_rf, rmse_finetune, rf_best_params = train_model(RandomForestRegressor(random_state=42), rf_params, X_train_num, y_train, X_test_num, y_test)

# --- Affichage des résultats ---
print("Random Forest Regression :")
print(f"RMSE brut           : {rmse_brut:.6f}")
print(f"RMSE normalisé      : {rmse_norm:.6f}")
print(f"RMSE feat select    : {rmse_feat:.6f}")
print(f"RMSE fine-tuning    : {rmse_finetune:.6f}")
print(f"Meilleurs hyperparamètres fine-tuning : {rf_best_params}")


Random Forest Regression :
RMSE brut           : 2.046145
RMSE normalisé      : 2.046145
RMSE feat select    : 2.110852
RMSE fine-tuning    : 2.029543
Meilleurs hyperparamètres fine-tuning : {'n_estimators': 400, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_features': 1.0, 'max_depth': 20}


In [13]:
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_squared_error
import numpy as np

# --- Colonnes numériques seulement ---
X_train_num = X_train.select_dtypes(include=[np.number])
X_test_num  = X_test.select_dtypes(include=[np.number])

# --- 1) RMSE brut ---
gb_brut = GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, random_state=42)
gb_brut.fit(X_train_num, y_train)
y_pred_brut = np.expm1(gb_brut.predict(X_test_num))
rmse_brut = np.sqrt(mean_squared_error(np.expm1(y_test), y_pred_brut))

# --- 2) RMSE normalisé (pas nécessaire mais pour homogénéité) ---
gb_norm = GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, random_state=42)
gb_norm.fit(X_train_num, y_train)
y_pred_norm = np.expm1(gb_norm.predict(X_test_num))
rmse_norm = np.sqrt(mean_squared_error(np.expm1(y_test), y_pred_norm))

# --- 3) RMSE feature selection ---
numeric_cols_train = X_train_num.columns
top_features = X_train_num.corrwith(y_train).abs().sort_values(ascending=False).index[:6].tolist()

gb_feat = GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, random_state=42)
gb_feat.fit(X_train_num[top_features], y_train)
y_pred_feat = np.expm1(gb_feat.predict(X_test_num[top_features]))
rmse_feat = np.sqrt(mean_squared_error(np.expm1(y_test), y_pred_feat))

# --- 4) RMSE fine-tuning rapide ---
gb_params = {
    'n_estimators': [100, 200, 300, 400],
    'learning_rate': [0.05, 0.1, 0.2],
    'max_depth': [3, 5, 7, 10],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'subsample': [0.6, 0.8, 1.0]
}

best_gb, rmse_finetune, gb_best_params = train_model(GradientBoostingRegressor(random_state=42), gb_params, X_train_num, y_train, X_test_num, y_test)

# --- Affichage des résultats ---
print("Gradient Boosting Regression :")
print(f"RMSE brut           : {rmse_brut:.6f}")
print(f"RMSE normalisé      : {rmse_norm:.6f}")
print(f"RMSE feat select    : {rmse_feat:.6f}")
print(f"RMSE fine-tuning    : {rmse_finetune:.6f}")
print(f"Meilleurs hyperparamètres fine-tuning : {gb_best_params}")


Gradient Boosting Regression :
RMSE brut           : 2.588214
RMSE normalisé      : 2.588214
RMSE feat select    : 2.625317
RMSE fine-tuning    : 1.829209
Meilleurs hyperparamètres fine-tuning : {'subsample': 1.0, 'n_estimators': 400, 'min_samples_split': 2, 'min_samples_leaf': 4, 'max_depth': 7, 'learning_rate': 0.1}


In [14]:
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import mean_squared_error
import numpy as np

# --- Sélection des colonnes numériques ---
X_train_num = X_train.select_dtypes(include=[np.number])
X_test_num  = X_test.select_dtypes(include=[np.number])

# --- Target log-transformée ---
y_train_log = np.log1p(y_train)
y_test_log  = np.log1p(y_test)

# --- RMSE brute ---
gb_brut = GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, random_state=42)
gb_brut.fit(X_train_num, y_train_log)
y_pred_gb_brut = np.expm1(gb_brut.predict(X_test_num))
rmse_gb_brut = np.sqrt(mean_squared_error(y_test, y_pred_gb_brut))

# --- RMSE feature selection (top 6 features) ---
top_features = X_train_num.corrwith(y_train_log).abs().sort_values(ascending=False).index[:6].tolist()
gb_feat = GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, random_state=42)
gb_feat.fit(X_train_num[top_features], y_train_log)
y_pred_gb_feat = np.expm1(gb_feat.predict(X_test_num[top_features]))
rmse_gb_feat = np.sqrt(mean_squared_error(y_test, y_pred_gb_feat))

# --- Fine-tuning rapide ---
gb_params = {
    'n_estimators': [100, 200, 300, 400],
    'learning_rate': [0.05, 0.1, 0.2],
    'max_depth': [3, 5, 7, 10],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'subsample': [0.6, 0.8, 1.0]
}
gb_search = RandomizedSearchCV(GradientBoostingRegressor(random_state=42),
                               param_distributions=gb_params,
                               n_iter=20,
                               scoring='neg_root_mean_squared_error',
                               cv=3,
                               n_jobs=-1,
                               random_state=42)
gb_search.fit(X_train_num, y_train_log)
best_gb = gb_search.best_estimator_
y_pred_gb_finetune = np.expm1(best_gb.predict(X_test_num))
rmse_gb_finetune = np.sqrt(mean_squared_error(y_test, y_pred_gb_finetune))

print("Gradient Boosting Regression :")
print(f"RMSE brut        : {rmse_gb_brut:.6f}")
print(f"RMSE feat select : {rmse_gb_feat:.6f}")
print(f"RMSE fine-tuning : {rmse_gb_finetune:.6f}")
print("Meilleurs hyperparamètres GB :", gb_search.best_params_)


Gradient Boosting Regression :
RMSE brut        : 0.371661
RMSE feat select : 0.374569
RMSE fine-tuning : 0.266783
Meilleurs hyperparamètres GB : {'subsample': 1.0, 'n_estimators': 400, 'min_samples_split': 2, 'min_samples_leaf': 4, 'max_depth': 7, 'learning_rate': 0.1}


In [15]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import mean_squared_error
import numpy as np

# --- Sélection des colonnes numériques ---
X_train_num = X_train.select_dtypes(include=[np.number])
X_test_num  = X_test.select_dtypes(include=[np.number])

# --- Target log-transformée ---
y_train_log = np.log1p(y_train)
y_test_log  = np.log1p(y_test)

# --- RMSE brute ---
rf_brut = RandomForestRegressor(n_estimators=100, random_state=42)
rf_brut.fit(X_train_num, y_train_log)
y_pred_rf_brut = np.expm1(rf_brut.predict(X_test_num))
rmse_rf_brut = np.sqrt(mean_squared_error(y_test, y_pred_rf_brut))

# --- RMSE feature selection (top 6 features) ---
top_features = X_train_num.corrwith(y_train_log).abs().sort_values(ascending=False).index[:6].tolist()
rf_feat = RandomForestRegressor(n_estimators=100, random_state=42)
rf_feat.fit(X_train_num[top_features], y_train_log)
y_pred_rf_feat = np.expm1(rf_feat.predict(X_test_num[top_features]))
rmse_rf_feat = np.sqrt(mean_squared_error(y_test, y_pred_rf_feat))

# --- Fine-tuning rapide ---
rf_params = {
    'n_estimators': [100, 200, 300, 400, 500],
    'max_depth': [None, 5, 10, 20, 30],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': [0.6, 0.8, 1.0, None]
}
rf_search = RandomizedSearchCV(RandomForestRegressor(random_state=42),
                               param_distributions=rf_params,
                               n_iter=50,
                               scoring='neg_root_mean_squared_error',
                               cv=3,
                               n_jobs=-1,
                               random_state=42,
                               verbose=2)
rf_search.fit(X_train_num, y_train_log)
best_rf = rf_search.best_estimator_
y_pred_rf_finetune = np.expm1(best_rf.predict(X_test_num))
rmse_rf_finetune = np.sqrt(mean_squared_error(y_test, y_pred_rf_finetune))

print("Random Forest Regression :")
print(f"RMSE brut        : {rmse_rf_brut:.6f}")
print(f"RMSE feat select : {rmse_rf_feat:.6f}")
print(f"RMSE fine-tuning : {rmse_rf_finetune:.6f}")
print("Meilleurs hyperparamètres RF :", rf_search.best_params_)


Fitting 3 folds for each of 50 candidates, totalling 150 fits
Random Forest Regression :
RMSE brut        : 0.309618
RMSE feat select : 0.320199
RMSE fine-tuning : 0.300979
Meilleurs hyperparamètres RF : {'n_estimators': 500, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_features': 0.8, 'max_depth': 20}


In [17]:
import pandas as pd
import numpy as np

# Suppose que `best_gb` est ton Gradient Boosting fine-tuné sur log1p(Montant)
# et que `X_train_num` contient les colonnes numériques utilisées pour l'entraînement

# --- Saisie des features par l'utilisateur ---
print("Saisir les valeurs des features :")
feature_values = {}
for col in X_train_num.columns:
    val = input(f"{col} : ")
    try:
        val = float(val)
    except:
        print("Valeur non numérique détectée, mise à 0 par défaut")
        val = 0.0
    feature_values[col] = val

# Création d'un DataFrame avec une seule ligne
df_input = pd.DataFrame([feature_values], columns=X_train_num.columns)

# Prédiction avec le modèle Gradient Boosting (log-transform)
y_pred_log = best_gb.predict(df_input)

# Retour à l'échelle originale
y_pred_real = np.expm1(y_pred_log)

print(f"\nMontant prédit : {y_pred_real[0]:.4f}")


Saisir les valeurs des features :

Montant prédit : 0.9335


In [23]:
from sklearn.ensemble import GradientBoostingRegressor

# Vérification du type de modèle
if isinstance(best_gb, GradientBoostingRegressor):
    print("✅ Le modèle utilisé est bien Gradient Boosting Regressor.")
else:
    print("❌ Attention : le modèle utilisé n'est pas Gradient Boosting Regressor.")


✅ Le modèle utilisé est bien Gradient Boosting Regressor.
